Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\24pasos_gru_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 7)
Dimensiones de Y: (43765, 1)


In [9]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 7)
Las dimensiones de testX son:  (8797, 12, 7)
Las dimensiones de valX son:  (4333, 12, 7)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

193/193 - 35s - 179ms/step - ia: 0.2268 - loss: 1.1278 - mae: 0.7959 - rmse: 1.0390 - smape: 1.5621 - val_ia: 0.2702 - val_loss: 0.4930 - val_mae: 0.5324 - val_rmse: 0.6198 - val_smape: 1.4117

Epoch 2/128                                           

193/193 - 6s - 29ms/step - ia: 0.3199 - loss: 1.0091 - mae: 0.7491 - rmse: 0.9889 - smape: 1.3772 - val_ia: 0.2683 - val_loss: 0.4766 - val_mae: 0.5109 - val_rmse: 0.6030 - val_smape: 1.2659

Epoch 3/128                                           

193/193 - 5s - 27ms/step - ia: 0.3519 - loss: 0.9828 - mae: 0.7373 - rmse: 0.9717 - smape: 1.3283 - val_ia: 0.2677 - val_loss: 0.4725 - val_mae: 0.5075 - val_rmse: 0.5998 - val_smape: 1.2624

Epoch 4/128                                           

193/193 - 6s - 31ms/step - ia: 0.3612 - loss: 0.9610 - mae: 0.7296 - rmse: 0.9641 - smape: 1.3144 - val_ia: 0.2654 - val_loss: 0.4681 - val_mae: 0.5063 - val_rmse: 0.5987 - val_smape: 1.2658

Epoch 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                         

25/25 - 41s - 2s/step - ia: 0.1718 - loss: 1.0583 - mae: 0.7617 - rmse: 1.0135 - smape: 1.6221 - val_ia: 0.2563 - val_loss: 0.4614 - val_mae: 0.4972 - val_rmse: 0.6702 - val_smape: 1.1996

Epoch 2/16                                                                         

25/25 - 4s - 167ms/step - ia: 0.4099 - loss: 0.9199 - mae: 0.7084 - rmse: 0.9427 - smape: 1.2195 - val_ia: 0.2566 - val_loss: 0.4667 - val_mae: 0.5002 - val_rmse: 0.6713 - val_smape: 1.2104

Epoch 3/16                                                                         

25/25 - 4s - 156ms/step - ia: 0.4384 - loss: 0.8616 - mae: 0.6768 - rmse: 0.9289 - smape: 1.1564 - val_ia: 0.2767 - val_loss: 0.4910 - val_mae: 0.5313 - val_rmse: 0.6903 - val_smape: 1.3211

Epoch 4/16                                                                         

25/25 - 4s - 160ms/step - ia: 0.4576 - loss: 0.8210 - mae: 0.6623 - rmse: 0.8889 - smape:

...

Epoch 1/16                                                                          

49/49 - 24s - 497ms/step - ia: 0.3819 - loss: 0.9494 - mae: 0.7215 - rmse: 0.9806 - smape: 1.2804 - val_ia: 0.3128 - val_loss: 0.4669 - val_mae: 0.5076 - val_rmse: 0.6401 - val_smape: 1.2233

Epoch 2/16                                                                          

49/49 - 2s - 39ms/step - ia: 0.4219 - loss: 0.8949 - mae: 0.7036 - rmse: 0.9384 - smape: 1.2237 - val_ia: 0.3031 - val_loss: 0.4642 - val_mae: 0.5043 - val_rmse: 0.6360 - val_smape: 1.2333

Epoch 3/16                                                                          

49/49 - 2s - 36ms/step - ia: 0.4336 - loss: 0.8759 - mae: 0.6882 - rmse: 0.9312 - smape: 1.2035 - val_ia: 0.2996 - val_loss: 0.4526 - val_mae: 0.4971 - val_rmse: 0.6288 - val_smape: 1.2134

Epoch 4/16                                                                          

49/49 - 2s - 34ms/step - ia: 0.4431 - loss: 0.8541 - mae: 0.6783 - rmse: 0.9122 - sm

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



25/25 - 14s - 557ms/step - ia: 0.3934 - loss: 1.3966 - mae: 0.7916 - rmse: 1.1744 - smape: 1.1529 - val_ia: 0.3228 - val_loss: 0.6164 - val_mae: 0.5273 - val_rmse: 0.7508 - val_smape: 0.9771

Epoch 2/128                                                                        

25/25 - 1s - 21ms/step - ia: 0.3864 - loss: 1.4090 - mae: 0.7945 - rmse: 1.1759 - smape: 1.1557 - val_ia: 0.3224 - val_loss: 0.6157 - val_mae: 0.5270 - val_rmse: 0.7504 - val_smape: 0.9772

Epoch 3/128                                                                        

25/25 - 0s - 16ms/step - ia: 0.3875 - loss: 1.4105 - mae: 0.7945 - rmse: 1.1774 - smape: 1.1547 - val_ia: 0.3220 - val_loss: 0.6150 - val_mae: 0.5267 - val_rmse: 0.7500 - val_smape: 0.9774

Epoch 4/128                                                                        

25/25 - 0s - 15ms/step - ia: 0.3899 - loss: 1.4031 - mae: 0.7949 - rmse: 1.2007 - smape: 1.1629 - val_ia: 0.3215 - val_loss: 0.6142 - val_mae: 0.5265 - val_rmse: 0.7496 - va

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/32                                                                         

97/97 - 29s - 296ms/step - ia: 0.3969 - loss: 1.9304 - mae: 0.9558 - rmse: 1.3837 - smape: 1.1535 - val_ia: 0.3159 - val_loss: 0.7316 - val_mae: 0.5695 - val_rmse: 0.7302 - val_smape: 0.9629

Epoch 2/32                                                                         

97/97 - 2s - 24ms/step - ia: 0.3600 - loss: 1.5448 - mae: 0.8473 - rmse: 1.2251 - smape: 1.2430 - val_ia: 0.2806 - val_loss: 0.5659 - val_mae: 0.5157 - val_rmse: 0.6550 - val_smape: 1.0372

Epoch 3/32                                                                         

97/97 - 2s - 25ms/step - ia: 0.3204 - loss: 1.3516 - mae: 0.8110 - rmse: 1.1567 - smape: 1.3278 - val_ia: 0.2648 - val_loss: 0.5235 - val_mae: 0.5295 - val_rmse: 0.6559 - val_smape: 1.2577

Epoch 4/32                                                                         

97/97 - 2s - 21ms/step - ia: 0.2685 - loss: 1.3047 - mae: 0.8269 - rmse: 1.1325 - smape:

In [16]:
print(best)

{'activation': 1, 'batch': 3, 'dropout': 0.2, 'epochs': 1, 'layers': 1.0, 'learning_rate': 0.002508532347133338, 'units': 4}
